# Library RNN baseline notebook

This notebook builds a standard-library recurrent baseline using PyTorch `nn.RNN` to compare the C implementation against a common framework-based approach.

## Baseline Intent

- This notebook is a methodological baseline, not a direct line-by-line mirror of the C code.
- It keeps the same dataset, sequence length, 80/20 split, train-only normalization, and evaluation metrics so the comparison stays as fair as possible.
- Core settings: input size 41, hidden size 64, output size 1, sequence length 8, batch size 16, learning rate 0.001, gradient clip 5.0, and 10 epochs.
- Using SGD to better match the C update rule.
- Added gradient clipping with the same clip value used in C.
- Overrode PyTorch default initialization with uniform `[-0.1, 0.1]` for weights and zero bias.


In [10]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

INPUT_SIZE = 41
HIDDEN_SIZE = 64
OUTPUT_SIZE = 1
SEQ_LEN = 8
BATCH_SIZE = 16
LEARNING_RATE = 0.001
GRAD_CLIP = 5.0
EPSILON = 1e-7
NUM_EPOCHS = 10
SEED = 42


DATASET_PATH = r'C:\Users\achiu\Documents\Thesis\KDD_Dataset\KDD_processed_dataset.csv'
if DATASET_PATH is None:
    raise FileNotFoundError('Could not find file.')

print('Using dataset:', DATASET_PATH)
np.set_printoptions(suppress=True, precision=6)
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using dataset: C:\Users\achiu\Documents\Thesis\KDD_Dataset\KDD_processed_dataset.csv
Using device: cuda


In [11]:
df = pd.read_csv(DATASET_PATH)
expected_cols = [str(i) for i in range(INPUT_SIZE)] + ['label']
if list(df.columns) != expected_cols:
    raise ValueError(f'Unexpected columns. Expected {expected_cols[:5]}...{expected_cols[-3:]}, got {df.columns.tolist()[:5]}...')

features = df[[str(i) for i in range(INPUT_SIZE)]].to_numpy(dtype=np.float32)
labels = (df['label'].to_numpy(dtype=np.float32) > 0.5).astype(np.int64)

print('Rows loaded:', len(df))
print('Feature shape:', features.shape)
print('Label counts:', dict(zip(*np.unique(labels, return_counts=True))))

Rows loaded: 118939
Feature shape: (118939, 41)
Label counts: {0: 116521, 1: 2418}


In [12]:
def build_sequences(features, labels, seq_len=SEQ_LEN):
    n_rows = len(features)
    n_seq = n_rows - seq_len + 1
    if n_seq <= 0:
        raise ValueError(f'Not enough rows to build sequences of length {seq_len}')
    X = np.empty((n_seq, seq_len, features.shape[1]), dtype=np.float32)
    y = np.empty((n_seq,), dtype=np.int64)
    for start in range(n_seq):
        X[start] = features[start:start + seq_len]
        y[start] = labels[start + seq_len - 1]
    return X, y

X_all, y_all = build_sequences(features, labels)
print('Built sequences:', len(X_all))

Built sequences: 118932


In [13]:
# Match the C data pipeline: 80/20 split, then train-only normalization

# Shuffle
rng = np.random.RandomState(SEED)
perm = rng.permutation(len(X_all))
X_all = X_all[perm]
y_all = y_all[perm]

train_size = int(0.8 * len(X_all))
test_size = len(X_all) - train_size
X_train, y_train = X_all[:train_size], y_all[:train_size]
X_test, y_test = X_all[train_size:], y_all[train_size:]

mean = X_train.mean(axis=(0, 1), dtype=np.float64).astype(np.float32)
var = X_train.astype(np.float64).var(axis=(0, 1)).astype(np.float32)
var = np.maximum(var, 1e-12)
std = np.sqrt(var).astype(np.float32)

X_train = ((X_train - mean) / (std + EPSILON)).astype(np.float32)
X_test = ((X_test - mean) / (std + EPSILON)).astype(np.float32)

train_neg = int(np.sum(y_train == 0))
train_pos = int(np.sum(y_train == 1))
pos_weight = np.float32(train_neg / train_pos) if train_pos > 0 else np.float32(1.0)

print('Train sequences:', train_size)
print('Test sequences:', test_size)
print('Train class counts -> neg:', train_neg, 'pos:', train_pos, '| pos_weight:', float(pos_weight))

Train sequences: 95145
Test sequences: 23787
Train class counts -> neg: 93179 pos: 1966 | pos_weight: 47.39521789550781


In [14]:
train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train.astype(np.float32)))
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test.astype(np.float32)))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [15]:
class LibraryRNNBaseline(nn.Module):
    def __init__(self, input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            nonlinearity='tanh',
            batch_first=True,
            num_layers=1,
            bias=True,
        )
        self.out = nn.Linear(hidden_size, 1)
        self.reset_parameters_like_c()

    def reset_parameters_like_c(self):
        with torch.no_grad():
            for name, param in self.rnn.named_parameters():
                if 'weight' in name:
                    param.uniform_(-0.1, 0.1)
                elif 'bias' in name:
                    param.zero_()
            self.out.weight.uniform_(-0.1, 0.1)
            self.out.bias.zero_()

    def forward(self, x):
        rnn_out, _ = self.rnn(x)
        last_hidden = rnn_out[:, -1, :]
        logits = self.out(last_hidden).squeeze(-1)
        return logits

model = LibraryRNNBaseline().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(pos_weight), device=device))
criterion_eval = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
model

LibraryRNNBaseline(
  (rnn): RNN(41, 64, batch_first=True)
  (out): Linear(in_features=64, out_features=1, bias=True)
)

In [16]:
def safe_div(num, den):
    return 0.0 if den == 0 else num / den

def evaluate_model(model, loader):
    model.eval()
    total_loss = 0.0
    all_scores = []
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion_eval(logits, yb)
            total_loss += float(loss.item()) * len(xb)

            scores = torch.sigmoid(logits)
            preds = (scores >= 0.5).to(torch.int64)
            all_scores.append(scores.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_targets.append(yb.cpu().numpy().astype(np.int64))

    scores = np.concatenate(all_scores)
    preds = np.concatenate(all_preds)
    targets = np.concatenate(all_targets)

    tp = int(np.sum((targets == 1) & (preds == 1)))
    tn = int(np.sum((targets == 0) & (preds == 0)))
    fp = int(np.sum((targets == 0) & (preds == 1)))
    fn = int(np.sum((targets == 1) & (preds == 0)))

    acc = 100.0 * safe_div(int(np.sum(preds == targets)), len(targets))
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2.0 * precision * recall, precision + recall)
    avg_loss = safe_div(total_loss, len(targets))

    return {
        'loss': avg_loss,
        'acc': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp,
        'scores': scores,
        'preds': preds,
        'targets': targets,
    }

In [17]:
def train_and_evaluate_library_rnn(model, train_loader, test_loader, num_epochs=NUM_EPOCHS):
    history = []
    prediction_rows = []
    wall_start = time.time()

    for epoch in range(num_epochs):
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_count = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_value_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            with torch.no_grad():
                scores = torch.sigmoid(logits)
                preds = (scores >= 0.5).to(torch.int64)
                train_correct += int((preds == yb.to(torch.int64)).sum().item())
                train_loss_sum += float(loss.item()) * len(xb)
                train_count += len(xb)

        train_acc = 100.0 * safe_div(train_correct, train_count)
        avg_train_loss = safe_div(train_loss_sum, train_count)

        test_metrics = evaluate_model(model, test_loader)
        elapsed_wall = time.time() - wall_start

        for i in range(min(10, len(test_metrics['targets']))):
            prediction_rows.append({
                'epoch': epoch + 1,
                'sample': i + 1,
                'true_label': int(test_metrics['targets'][i]),
                'pred_label': int(test_metrics['preds'][i]),
                'pred_score': float(test_metrics['scores'][i]),
            })

        row = {
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'train_acc': train_acc,
            'test_loss': test_metrics['loss'],
            'test_acc': test_metrics['acc'],
            'precision': test_metrics['precision'],
            'recall': test_metrics['recall'],
            'f1': test_metrics['f1'],
            'tn': test_metrics['tn'],
            'fp': test_metrics['fp'],
            'fn': test_metrics['fn'],
            'tp': test_metrics['tp'],
            'elapsed_sec': elapsed_wall,
        }
        history.append(row)

        print(f"Epoch {epoch + 1} TRAIN complete | Samples {train_count} | Avg Loss {avg_train_loss:.6f} | Accuracy {train_acc:.2f}%")
        print(f"Epoch {epoch + 1} TEST complete | Samples {len(test_metrics['targets'])} | Avg Loss {test_metrics['loss']:.6f} | Accuracy {test_metrics['acc']:.2f}% | Precision {test_metrics['precision']:.4f} | Recall {test_metrics['recall']:.4f} | F1 {test_metrics['f1']:.4f} | Elapsed {elapsed_wall:.2f} sec")
        print('Confusion Matrix (TEST):')
        print(' Pred 0 Pred 1')
        print(f"True 0 {test_metrics['tn']:8d} {test_metrics['fp']:8d}")
        print(f"True 1 {test_metrics['fn']:8d} {test_metrics['tp']:8d}\n")

    return pd.DataFrame(history), pd.DataFrame(prediction_rows)
    #redictions_df.head(20)

In [18]:
history_df, predictions_df = train_and_evaluate_library_rnn(model, train_loader, test_loader, num_epochs=NUM_EPOCHS)
history_df

Epoch 1 TRAIN complete | Samples 95145 | Avg Loss 0.262694 | Accuracy 96.05%
Epoch 1 TEST complete | Samples 23787 | Avg Loss 0.050563 | Accuracy 98.73% | Precision 0.6019 | Recall 0.9735 | F1 0.7439 | Elapsed 24.07 sec
Confusion Matrix (TEST):
 Pred 0 Pred 1
True 0    23044      291
True 1       12      440

Epoch 2 TRAIN complete | Samples 95145 | Avg Loss 0.109575 | Accuracy 98.53%
Epoch 2 TEST complete | Samples 23787 | Avg Loss 0.040859 | Accuracy 98.99% | Precision 0.6577 | Recall 0.9779 | F1 0.7865 | Elapsed 47.21 sec
Confusion Matrix (TEST):
 Pred 0 Pred 1
True 0    23105      230
True 1       10      442

Epoch 3 TRAIN complete | Samples 95145 | Avg Loss 0.088709 | Accuracy 98.74%
Epoch 3 TEST complete | Samples 23787 | Avg Loss 0.033037 | Accuracy 99.10% | Precision 0.6842 | Recall 0.9779 | F1 0.8051 | Elapsed 70.46 sec
Confusion Matrix (TEST):
 Pred 0 Pred 1
True 0    23131      204
True 1       10      442

Epoch 4 TRAIN complete | Samples 95145 | Avg Loss 0.074480 | Accura

,epoch,train_loss,train_acc,test_loss,test_acc,precision,recall,f1,tn,fp,fn,tp,elapsed_sec
0,1,0.262694,96.046035,0.050563,98.726195,0.601915,0.973451,0.743872,23044,291,12,440,24.066752
1,2,0.109575,98.525409,0.040859,98.991046,0.657738,0.977876,0.786477,23105,230,10,442,47.209313
2,3,0.088709,98.736665,0.033037,99.100349,0.684211,0.977876,0.805100,23131,204,10,442,70.464651
3,4,0.074480,98.880656,0.026843,99.171817,0.702703,0.977876,0.817761,23148,187,10,442,93.979465
4,5,0.064821,98.956330,0.023182,99.197040,0.708800,0.980088,0.822656,23153,182,9,443,117.962789
5,6,0.058110,99.001524,0.020343,99.222264,0.715670,0.980088,0.827264,23159,176,9,443,141.650006
6,7,0.052972,99.045667,0.018286,99.247488,0.722675,0.980088,0.831925,23165,170,9,443,163.308572
7,8,0.048885,99.072994,0.016753,99.335772,0.747475,0.982301,0.848948,23185,150,8,444,184.584230
8,9,0.045490,99.147617,0.015476,99.445075,0.781690,0.982301,0.870588,23211,124,8,444,208.588964
9,10,0.042543,99.231699,0.014350,99.503931,0.801444,0.982301,0.882704,23225,110,8,444,232.386632
